In [1]:
import os
import re
import json
import time
import tiktoken

from IPython.core.display_functions import clear_output
from dotenv import load_dotenv
from pathlib import Path
from tuutrag.data import D
from tuutrag.types import *
from tuutrag.prompt import create_batch_request
from tuutrag.prompts.manager import load_prompt
from tuutrag.utils import count_batch_request_tokens
from openai import OpenAI
from tqdm import tqdm

In [ ]:
# %load_ext autoreload
# %autoreload 2

In [2]:
load_dotenv()

OPENAI_MODEL: str = "gpt-4.1-mini"

openai_key: str | None = os.getenv("OPENAI_API_KEY")
if not openai_key:
    raise ValueError("OPENAI_API_KEY is not set in environment variables.")

openai_client: OpenAI = OpenAI(api_key=openai_key)

In [ ]:
def get_branches(conn):
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM branches")
    return cursor.fetchall()

In [ ]:
def get_branch_summaries() -> list[BranchSummary]:
    """Load branch summaries from JSON file and return as a list of BranchSummary dicts."""


In [3]:
with open(D.find("branch_summaries.json"), "r", encoding="utf-8") as f:
    all_chunks: list[BranchSummary] = json.load(f)

seen: set[str] = set()
unique_branches: list[BranchSummary] = []
for obj in all_chunks:
    if obj["uuid"] not in seen:
        seen.add(obj["uuid"])
        unique_branches.append(obj)

In [4]:
SCHEME: str = "o200k_base"
enc: tiktoken.Encoding = tiktoken.get_encoding(SCHEME)

total_tokens: int = sum(len(enc.encode(obj["text"])) for obj in unique_branches)
print(f"{total_tokens:,}")

5,661,241


In [5]:
artifact_map: dict[str, list] = {}

for obj in unique_branches:
    artifact_id = obj["uuid"].rsplit(".", 1)[0]
    if artifact_id not in artifact_map:
        artifact_map[artifact_id] = [{
            "uuid": obj["uuid"],
            "text": obj["text"]
        }]
    else:
        artifact_map[artifact_id].append({
            "uuid": obj["uuid"],
            "text": obj["text"]
        })
all_requests: list[BatchRequest] = [
    create_batch_request(
        custom_id=key,
        model=OPENAI_MODEL,
        **load_prompt("summary.j2", text="\n".join(f"{count}. {obj['text']}\n" for count, obj in enumerate(value, start=1)))
    )
    for key, value in tqdm(artifact_map.items(), desc="Creating batch requests")
]

token_counts: list[int] = [
    count_batch_request_tokens(
        enc=enc,
        payload=req
    ) for req in tqdm(all_requests, desc="Counting Total Tokens")
]



Counting Total Tokens: 100%|██████████| 453/453 [00:04<00:00, 107.28it/s]


In [6]:
print(len(all_requests))
print (f"Total tokens across all requests: {sum(token_counts):,}")

453
Total tokens across all requests: 5,820,666


In [ ]:
# Load already-processed UUIDs from branch_summaries.json
with open(D.find("branch_summaries.json"), "r", encoding="utf-8") as f:
    processed_ids: set[str] = {
        ".".join(obj["uuid"].split(".")[:2]) for obj in json.load(f)
    }

# Filter to only unprocessed requests
pending_requests: list[BatchRequest] = [
    req for req in all_requests if req["custom_id"] not in processed_ids
]
pending_token_counts: list[int] = [
    token_counts[i] for i, req in enumerate(all_requests)
    if req["custom_id"] not in processed_ids
]

print(f"Total requests    : {len(all_requests):,}")
print(f"Already processed : {len(processed_ids):,}")
print(f"Pending           : {len(pending_requests):,}")

In [ ]:
all_requests = pending_requests
token_counts = pending_token_counts

In [7]:
MAX_TOKENS_PER_BATCH: int = 1_000_000  # OpenAI batch limit (w/ safety padding)

batches: list[list[BatchRequest]] = []
current_batch: list[BatchRequest] = []
current_tokens: int = 0

for req, req_tokens in zip(all_requests, token_counts):
    if current_tokens + req_tokens > MAX_TOKENS_PER_BATCH and current_batch:
        batches.append(current_batch)
        current_batch = []
        current_tokens = 0
    current_batch.append(req)
    current_tokens += req_tokens

if current_batch:
    batches.append(current_batch)

print(f"Total requests : {len(all_requests):,}")
print(f"Total batches  : {len(batches):,}")
for i, batch in enumerate(batches):
    batch_tokens = sum(token_counts[all_requests.index(req)] for req in batch)
    print(f"  Batch {i + 1}: {len(batch):,} requests | {batch_tokens:,} tokens")

Total requests : 453
Total batches  : 7
  Batch 1: 4 requests | 832,537 tokens
  Batch 2: 5 requests | 930,106 tokens
  Batch 3: 5 requests | 940,781 tokens
  Batch 4: 5 requests | 949,331 tokens
  Batch 5: 284 requests | 988,022 tokens
  Batch 6: 75 requests | 997,886 tokens
  Batch 7: 75 requests | 182,003 tokens


In [8]:
POLL_INTERVAL: int = 30
RETRY_WAIT:    int = 120
MAX_RETRIES:   int = 5

metadata_path: Path = D.artifact_batches / "batch_metadata.json"
master_file:   Path = D.processed / "artifact_summaries.json"

In [9]:
def _save_metadata(metadata: list[dict]) -> None:
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

def _get_metadata() -> list[dict]:
    with open(metadata_path, "r", encoding="utf-8") as f:
        return json.load(f)

def _remove_from_metadata(batch_id: str) -> None:
    metadata = _get_metadata()
    _save_metadata([m for m in metadata if m.get("id") != batch_id])

In [10]:
def _append_branches(output_text: str) -> int:
    """Parse <|> delimited LLM output and append leaves to master file. Returns count added."""
    responses: list[dict] = [
        json.loads(line)
        for line in output_text.strip().split("\n")
        if line.strip()
    ]

    try:
       with open(master_file, "r", encoding="utf-8") as f:
            existing: list[dict] = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        existing = []

    added: int = 0
    for response in responses:
        res_custom_id: str = response["custom_id"]
        res_content: str   = response["response"]["body"]["choices"][0]["message"]["content"]
        summaries: list[str]  = [summary.strip() for summary in res_content.split("<|>") if summary.strip()]
        for i, summary in enumerate(summaries, start=1):
            existing.append({"uuid": f"{res_custom_id}", "text": summary})
            added += 1

    with open(master_file, "w", encoding="utf-8") as f:
        json.dump(existing, f, indent=2, ensure_ascii=False)

    return added

In [11]:
def _resubmit(old_batch_id: str) -> str:
    """Resubmit a failed batch from its original .jsonl and update metadata. Returns new batch_id."""
    metadata  = _get_metadata()
    entry     = next(m for m in metadata if m["id"] == old_batch_id)

    with open(D.artifact_batches() / entry["file_name"], "rb") as f:
        batch_file = openai_client.files.create(file=f, purpose="batch")

    new_job = openai_client.batches.create(
        input_file_id=batch_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )

    entry["id"] = new_job.id
    _save_metadata(metadata)

    return new_job.id

In [ ]:
pattern = re.compile(r"^summary_batch_\d+\.jsonl$")

# Delete matched .jsonl files locally
deleted_local = [
    f for f in metadata_path.parent.iterdir()
    if f.is_file() and pattern.match(f.name) and not f.unlink()
]
print(f"Deleted locally  : {len(deleted_local):,} .jsonl files")

# Delete input files and cancel batch jobs on OpenAI
deleted_files  = 0
cancelled_jobs = 0

try:
    metadata = _get_metadata()
    for entry in metadata:
        # Delete uploaded input file
        file_id = entry.get("input_file_id")
        if file_id:
            try:
                openai_client.files.delete(file_id)
                deleted_files += 1
                print(f"  Deleted file      : {file_id}  ({entry['file_name']})")
            except Exception as e:
                print(f"  Could not delete file {file_id}: {e}")

        # Cancel batch job
        batch_id = entry.get("id")
        if batch_id:
            try:
                openai_client.batches.cancel(batch_id)
                cancelled_jobs += 1
                print(f"  Cancelled batch   : {batch_id}")
            except Exception as e:
                print(f"  Could not cancel batch {batch_id}: {e}")

except FileNotFoundError:
    print("  No batch_metadata.json found — skipping remote cleanup.")

print(f"Deleted files    : {deleted_files:,}")
print(f"Cancelled jobs   : {cancelled_jobs:,}")

# Delete batch_metadata.json
if metadata_path.exists():
    metadata_path.unlink()
    print("Deleted          : batch_metadata.json")

In [12]:
batch_file_ids: list[dict] = []
batch_metadata: list[dict] = []

for idx, batch in enumerate(tqdm(batches, desc="Uploading batch files")):
    file_name: str = f"summary_batch_{idx}.jsonl"

    with open(metadata_path.parent / file_name, "w", encoding="utf-8") as f:
        for request in batch:
            f.write(json.dumps(request) + "\n")

    with open(metadata_path.parent / file_name, "rb") as f:
        batch_file = openai_client.files.create(file=f, purpose="batch")

    batch_file_ids.append({
        "file_name": file_name,
        "input_file_id": batch_file.id,
    })

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(batch_file_ids, f, indent=2)

print(f"\nUploaded {len(batch_file_ids):,} files. Ready to submit jobs in Monitor cell.")

Uploading batch files: 100%|██████████| 7/7 [00:13<00:00,  1.97s/it]


Uploaded 7 files. Ready to submit jobs in Monitor cell.


In [13]:
print(f"Submitting and processing {len(batch_file_ids)} batch(es) one at a time.\n")

for file_entry in batch_file_ids:
    file_name: str = file_entry["file_name"]
    input_file_id: str = file_entry["input_file_id"]
    retries: int = 0

    batch_job = openai_client.batches.create(
        input_file_id=input_file_id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )
    batch_id: str = batch_job.id

    metadata = _get_metadata()
    for m in metadata:
        if m["input_file_id"] == input_file_id:
            m.update({"id": batch_id, **batch_job.model_dump()})
    _save_metadata(metadata)

    print(f"{file_name} | submitted: {batch_id}")

    while True:
        job = openai_client.batches.retrieve(batch_id)
        status = job.status

        if status == "completed":
            output = openai_client.files.content(job.output_file_id)
            added = _append_branches(output.text)
            _remove_from_metadata(batch_id)
            print(f"Complete - {added} leaves added | {len(_get_metadata())} remaining\n")
            break

        elif status == "failed":
            errors = job.errors.data if job.errors else []
            token_limit = any(
                "enqueued token limit" in (e.message or "").lower()
                for e in errors
            )
            if token_limit and retries < MAX_RETRIES:
                retries += 1
                print(f"Token limit hit - waiting {RETRY_WAIT}s then resubmitting (attempt {retries}/{MAX_RETRIES})")
                time.sleep(RETRY_WAIT)
                batch_id = _resubmit(batch_id)
                print(f"Resubmitted as {batch_id}")
            elif retries >= MAX_RETRIES:
                print(f"Max retries ({MAX_RETRIES}) reached - skipping.\n")
                _remove_from_metadata(batch_id)
                break
            else:
                messages = [e.message for e in errors]
                print(f"Failed: {messages} - skipping.\n")
                _remove_from_metadata(batch_id)
                break

        elif status in ("cancelling", "cancelled", "expired"):
            print(f"Status '{status}' - skipping.\n")
            _remove_from_metadata(batch_id)
            break

        else:
            completed = job.request_counts.completed
            total = job.request_counts.total
            clear_output(wait=True)
            print(f"{file_name} | {batch_id}")
            print(f"[{status}] {completed}/{total} - next check in {POLL_INTERVAL}s...")
            time.sleep(POLL_INTERVAL)

print("All batches processed.")


summary_batch_6.jsonl | batch_69b3bebd36fc81908df97f14c9fdd4f8
[in_progress] 73/75 - next check in 30s...
Complete - 75 leaves added | 0 remaining

All batches processed.


In [14]:
def count_branch_summary_tokens() -> None:
    """Count the total tokens across all 'text' fields in branch_summaries.json."""
    enc: tiktoken.Encoding = tiktoken.get_encoding(SCHEME)

    with open(D.processed / "artifact_summaries.json", "r", encoding="utf-8") as f:
        data: list[dict] = json.load(f)

    total_tokens: int = sum(len(enc.encode(obj["text"])) for obj in data)

    print(f"{total_tokens:,}")

In [15]:
count_branch_summary_tokens()

64,882
